[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onuralpszr/litert-llm-cookbook/blob/main/colab/14_manual_tool_calling.ipynb)

# Example 14: Manual Tool Calling

Set automatic_tool_calling=False so the model returns tool call specifications instead of executing them. The application inspects each call, dispatches it manually, and feeds results back.

In [ ]:
!pip install litert-lm-api-nightly litert-lm -q

In [ ]:
import os
import subprocess

model_url = 'https://huggingface.co/litert-community/gemma-4-E4B-it-litert-lm/resolve/main/gemma-4-E4B-it.litertlm?download=true'
output_path = '/content/gemma-4-E4B-it.litertlm'

if not os.path.exists(output_path):
    subprocess.run(['curl', '-L', model_url, '-o', output_path], check=True)
    print(f'Model downloaded to {output_path}')
else:
    print(f'Model already exists at {output_path}')

In [ ]:
import json
from typing import Any

import litert_lm

MODEL_PATH = "/content/gemma-4-E4B-it.litertlm"

litert_lm.set_min_log_severity(litert_lm.LogSeverity.ERROR)


def _clean(value: str) -> str:
    return value.replace('<|"|>', "").strip()


def get_stock_price(ticker: str) -> float:
    """Returns the current stock price for a ticker symbol.

    Args:
        ticker: The stock ticker symbol. Must be one of: AAPL, GOOG, MSFT.
    """
    prices = {"AAPL": 189.50, "GOOG": 175.20, "MSFT": 415.80}
    return prices.get(_clean(ticker).upper(), -1.0)


def _dispatch(tool_call: dict[str, Any]) -> Any:
    fn = tool_call.get("function", {})
    name = fn.get("name", "")
    args = fn.get("arguments", {})
    print(f"  [app] executing {name}({args})")
    if name == "get_stock_price":
        return get_stock_price(**args)
    raise ValueError(f"Unknown tool: {name}")


with litert_lm.Engine(MODEL_PATH) as engine:
    with engine.create_conversation(
        tools=[get_stock_price],
        automatic_tool_calling=False,
    ) as conversation:
        query = "What are the current prices for AAPL and MSFT?"
        print(f"Q: {query}\n")

        response = conversation.send_message(query)

        answers = []
        for call in response.get("tool_calls", []):
            result = _dispatch(call)
            tool_message = {
                "role": "tool",
                "name": call["function"]["name"],
                "content": json.dumps(result),
            }
            tool_response = conversation.send_message(tool_message)
            text = tool_response.get("content", [{}])[0].get("text", "")
            if text:
                answers.append(text)

        print("\nA: " + "  ".join(answers))